# 00_b5 — Slovakia: municipality population by age

Part of the `00_b*` series that replaces the five legacy population tables in the local
`dbis` Postgres (read by `traintran/population.py` / `harmonize_population_lau2.ipynb`) with
newer national-statistics files, each brought to the **same age-band widths** and keyed on
`LAU CODE` from `EU-27-LAU-2025-NUTS-2024.xlsx`.

> **Working notebook — not for publication.** It connects to the local DB only to take the legacy
> table as a baseline; once the harmonised output exists, nothing downstream needs the DB.

| | dataset | location |
|---|---|---|
| legacy (DB) | `public.sk_population_age_groups_lau2_21` | local `dbis` Postgres |
| new (2025) | ŠÚ SR DATAcube `om7006rr` — *Vekové skupiny – SR-oblasť-kraj-okres-obec* (Age Groups – Municipalities), 31 Dec 2025, via the public API (<https://statdata.statistics.sk/public/api/dc/swagger-ui/index.html>; UI view <https://statdata.statistics.sk/public/ui/dc/domov/data-view/view/5a6aae41-c889-4055-8065-6e242444e106/om7006rr?lang=sk>) | `data/eu/ref-national-statistical-offices/sk/api/` |

**Legacy notes.** 5-year bands 0–4 … 60–64, 65+. Join: `soIdLau2` minus its `soIdLau1` prefix.

In [1]:
import os, sys

# Run from the project root so every relative path ('data/...', 'traintran/...') resolves the
# same whether the notebook is launched from notebooks/ or from the root.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: E:\notes\api_zssk\project


In [2]:
import os
from pathlib import Path

import pandas as pd
# import sqlalchemy as sa

import traintran  # noqa: F401  (fixes PROJ on import)
from traintran import population as pop
from traintran.config import default_config

COUNTRY = 'SK'
OUT_DIR = Path('data/eu/ref-national-statistical-offices') / COUNTRY.lower()
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = default_config()
# Database comparison disabled. The 2025 population comes from the statistical office's files below; the
# database held only the older 2021 tables used for a comparison that feeds no output. To re-enable it, set
# TRAINTRAN_PG_URL in the environment and uncomment these lines and the cells marked '[disabled: ...]'.
# engine = sa.create_engine(os.environ['TRAINTRAN_PG_URL'])
print('Country   :', COUNTRY)
print('Output dir:', OUT_DIR)
# print('Postgres  :', cfg.pg_url.rsplit('@', 1)[-1])

Country   : SK
Output dir: data\eu\ref-national-statistical-offices\sk


## 1. Legacy — local DB

The raw table(s) as they sit in `dbis`, geometry columns dropped. A parquet snapshot is written
next to the new source (skip-if-present) so the comparison in §3 can later run without the DB.

In [3]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# GEOM_COLS = {'geom', 'geojson', 'wkb_geometry', 'geometry'}
# LEGACY_TABLES = ['public.sk_population_age_groups_lau2_21']


# def read_table(qualified):
#     schema, table = qualified.split('.', 1)
#     table = table.strip('"')
#     cols = [c['name'] for c in sa.inspect(engine).get_columns(table, schema=schema)
#             if c['name'].lower() not in GEOM_COLS]
#     sel = ', '.join('"' + c.replace('"', '""') + '"' for c in cols)
#     return pd.read_sql(f'select {sel} from {schema}."{table}"', engine)


# legacy_raw = {}
# for t in LEGACY_TABLES:
#     legacy_raw[t] = df = read_table(t)
#     print(f'{t}: {len(df):,} rows x {df.shape[1]} cols')
#     print('  ', list(df.columns))

In [4]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# for t, df in legacy_raw.items():
#     stem = t.replace('.', '_').replace(chr(34), '')
#     snap = OUT_DIR / f'legacy_db_{stem}.parquet'
#     if snap.exists():
#         print('[exists]', snap)
#     else:
#         df.to_parquet(snap, index=False)
#         print('[wrote] ', snap, f'({len(df):,} rows)')

### 1a. Baseline — the current loader on the legacy table

`pop.LOADERS['SK']` is exactly what `harmonize_population_lau2.ipynb` runs today; its keyed and
banded result is what the 2025 data has to reproduce in structure (same `lau_code` coverage,
same harmonised bands).

In [5]:
lau = pop.load_workbook_lau(cfg.lau_file, countries=(COUNTRY,))
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# legacy = pop.harmonise_bands(pop.LOADERS[COUNTRY](engine, lau), COUNTRY)
# legacy['country'] = COUNTRY
# legacy['ref_year'] = 2021

# print(pop.match_report({COUNTRY: legacy}, lau).to_string(index=False))
# print()
# print('band_method  :', legacy.band_method.unique().tolist())
# print('band_complete:', int(legacy.band_complete.sum()), '/', len(legacy))
# print('pop_total    :', f'{legacy.pop_total.sum():,.0f}')
# print()
# print(legacy.head().to_string())

## 2. New (2025) — ŠÚ SR DATAcube API

Cube `om7006rr`, *Age Groups – Municipalities*. The manual DATAcube export gives municipality and
district **names only**, so the data is pulled from the public API instead
(<https://statdata.statistics.sk/public/api/dc/swagger-ui/index.html>), which knows each municipality's code.

How the API is used (read-only endpoints only):

1. `GET /public/om7006rr/info` → the cube's dimensions and their ids.
2. `GET /public/{dimensionId}/om7006rr/dimensionItemsInCube` → the items of a dimension:
   `acronym`, `nameSk`, `nameEn`. A municipality's acronym (`e_nuts15`) is `SK` + district + LAU
   code, e.g. `SK0237500011`, i.e. the legacy `soIdLau2` format.
3. `GET /opendata/cube/dataset/om7006rr/rows/{rows}/columns/{columns}/context/{context}?lang=SK&type=CSV`
   → the data. Each area lists dimensions separated by `~`, each as `acronym,ASC,item,item,…`
   (or `all`); a context dimension takes exactly one item.

**Selection.** Sex `TOTAL` · indicator `U_OM_0005` (population on 31 Dec) · unit `UNIT_PERS` ·
year `2025` · age `AGE_Y_LE14`, `AGE_Y15-64`, `AGE_Y_GE65` plus `TOTAL` as a check value.
The cube's other age items overlap these (0–17, 18+, 50+, 85+, …), so nothing else is requested.

**Chunking.** One call per district: the district goes in the context (`e_nuts14,ASC,<district>`)
and `e_nuts15,ASC,all` then returns only that district's municipalities. A single all-country
call with all ages ran for >10 min, while a district takes about a second. Each district's CSV is
cached in `sk/api/chunks/` and skipped if present, with a pause between calls.

**Names → codes.** Every output format (CSV/JSON/XML/RDF) labels rows by **name**, not acronym.
Names are unique across all 2 928 municipalities in the item list (duplicates already carry a
district suffix, e.g. *Hosťovce, okres Zlaté Moravce*), and the mapping is done within each
district and asserted complete.

In [6]:
import json
import time

import requests

API = 'https://statdata.statistics.sk/public/api/dc'
CUBE = 'om7006rr'
SELECTION = dict(sex='TOTAL', indic='U_OM_0005', unit='UNIT_PERS', year='2025')
AGES = ['TOTAL', 'AGE_Y_LE14', 'AGE_Y15-64', 'AGE_Y_GE65']
PAUSE_S = 1.5

API_DIR = OUT_DIR / 'api'
CHUNK_DIR = API_DIR / 'chunks'
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
http = requests.Session()


def get(url, tries=4, timeout=120):
    for k in range(tries):
        try:
            r = http.get(url, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if k == tries - 1:
                raise
            wait = 5 * 2 ** k
            print(f'  retry in {wait}s after {type(e).__name__}: {e}')
            time.sleep(wait)


def cached_json(name, url):
    p = API_DIR / name
    if p.exists():
        print('[exists]', p)
    else:
        p.write_text(get(url).text, encoding='utf-8')
        print('[wrote] ', p)
    return json.loads(p.read_text(encoding='utf-8'))


info = cached_json(f'{CUBE}_info.json', f'{API}/public/{CUBE}/info')
dims = {x['acronym']: x for x in info['dimensions']}
print(f"\n{info['nameEn']} ({info['nameSk'].strip()}); last data change {info['lastDataChange']}")
print('dimensions:', list(dims))

items = {a: cached_json(f'{CUBE}_items_{a}.json',
                        f"{API}/public/{dims[a]['id']}/{CUBE}/dimensionItemsInCube")
         for a in ['e_nuts14', 'e_nuts15', 'e_age']}

[exists] data\eu\ref-national-statistical-offices\sk\api\om7006rr_info.json

Age Groups - Municipalities (Vekové skupiny - SR-oblasť-kraj-okres-obec, m-v); last data change 2026-03-31T09:00:00+02:00
dimensions: ['e_sex', 'e_nuts14', 'e_indic', 'e_unit', 'e_nuts15', 'e_year', 'e_age']
[exists] data\eu\ref-national-statistical-offices\sk\api\om7006rr_items_e_nuts14.json
[exists] data\eu\ref-national-statistical-offices\sk\api\om7006rr_items_e_nuts15.json
[exists] data\eu\ref-national-statistical-offices\sk\api\om7006rr_items_e_age.json


In [7]:
# Municipalities = 12-char NUTS15 acronyms; their first 6 chars are the district.
muni = pd.DataFrame([i for i in items['e_nuts15'] if len(i['acronym']) == 12])
muni['district'] = muni['acronym'].str[:6]
assert muni['nameSk'].is_unique, 'municipality names repeat: name -> code mapping not safe'
districts = (pd.DataFrame(items['e_nuts14'])[['acronym', 'nameSk']]
             .rename(columns={'acronym': 'district', 'nameSk': 'districtName'}))
districts = districts[districts.district.isin(muni.district)].reset_index(drop=True)
districts['districtName'] = districts['districtName'].str.replace(r'^Okres ', '', regex=True)
age_names = {i['acronym']: i['nameSk'] for i in items['e_age']}
assert set(AGES) <= set(age_names), set(AGES) - set(age_names)
print(f'{len(muni):,} municipalities in {len(districts)} districts')

2,928 municipalities in 79 districts


In [8]:
def chunk_url(district):
    ctx = '~'.join([f"e_sex,ASC,{SELECTION['sex']}", f"e_indic,ASC,{SELECTION['indic']}",
                    f"e_unit,ASC,{SELECTION['unit']}", f"e_year,ASC,{SELECTION['year']}",
                    f'e_nuts14,ASC,{district}'])
    return (f'{API}/opendata/cube/dataset/{CUBE}/rows/e_nuts15,ASC,all'
            f"/columns/e_age,ASC,{','.join(AGES)}/context/{ctx}?lang=SK&type=CSV")


tag = f"{CUBE}_{SELECTION['year']}_{SELECTION['indic']}"
fetched = 0
t0 = time.time()
for k, dist in enumerate(districts.district, 1):
    p = CHUNK_DIR / f'{tag}_{dist}.csv'
    if p.exists():
        continue
    p.write_bytes(get(chunk_url(dist)).content)
    fetched += 1
    if fetched % 10 == 0:
        print(f'  {k}/{len(districts)} districts ({time.time() - t0:.0f}s)')
    time.sleep(PAUSE_S)
have = sorted(CHUNK_DIR.glob(f'{tag}_*.csv'))
print(f'[{"fetched " + str(fetched) if fetched else "exists"}] {len(have)}/{len(districts)} district chunks in {CHUNK_DIR}'
      + (f' ({time.time() - t0:.0f}s)' if fetched else ''))

[exists] 79/79 district chunks in data\eu\ref-national-statistical-offices\sk\api\chunks


### 2a. Raw chunks → one long table

Each chunk is `e_age, e_nuts15, hodnota` (age name, municipality name, value). The district comes
from the file name, the municipality code from the name within that district.

**The data labels are not always spelled like the item list.** Three municipalities carry a typo
in the data output only (e.g. *Mikušovce, okes Lučenec* vs *okres*). Names are therefore matched
exactly first; whatever is left in a district is paired one-to-one by close match (similarity ≥ 0.9),
and every such fix is printed. Javorina (military district) is in the item list but has no data.

In [9]:
import difflib

age_by_name = {v: k for k, v in age_names.items()}
parts, name_fixes = [], []
for p in have:
    dist = p.stem.rsplit('_', 1)[1]
    c = pd.read_csv(p, dtype=str, encoding='utf-8')
    assert list(c.columns) == ['e_age', 'e_nuts15', 'hodnota'], (p.name, list(c.columns))
    code_of = muni[muni.district == dist].set_index('nameSk')['acronym'].to_dict()

    left_data = sorted(set(c['e_nuts15']) - set(code_of))
    left_items = sorted(set(code_of) - set(c['e_nuts15']))
    for n in left_data:
        hit = difflib.get_close_matches(n, left_items, n=2, cutoff=0.9)
        assert len(hit) == 1, f'{dist}: cannot pair {n!r} with one of {left_items}'
        code_of[n] = code_of[hit[0]]
        left_items.remove(hit[0])
        name_fixes.append((dist, n, hit[0], code_of[n]))

    c['soIdLau2'] = c['e_nuts15'].map(code_of)
    assert c['soIdLau2'].notna().all()
    c['age'] = c['e_age'].map(age_by_name)
    assert c['age'].notna().all(), (dist, c.loc[c.age.isna(), 'e_age'].unique())
    c['soIdLau1'] = dist
    parts.append(c)

sk_raw = pd.concat(parts, ignore_index=True)
sk_raw['hodnota'] = pd.to_numeric(sk_raw['hodnota'], errors='raise')
print(f'{len(sk_raw):,} rows from {len(parts)} chunks; ages {sorted(sk_raw.age.unique())}')
print(f'\nname fixes (data label -> item list name): {len(name_fixes)}')
for f in name_fixes:
    print('  {}  {!r:<36} -> {!r}  {}'.format(*f))
sk_raw.head()

11,708 rows from 79 chunks; ages ['AGE_Y15-64', 'AGE_Y_GE65', 'AGE_Y_LE14', 'TOTAL']

name fixes (data label -> item list name): 3
  SK0326  'Mikušovce, okes Lučenec'            -> 'Mikušovce, okres Lučenec'  SK0326580309
  SK032D  'Lúčky, okres Žiar nad Hronon'       -> 'Lúčky, okres Žiar nad Hronom'  SK032D517046
  SK041A  'Kamienka, okres Stará Lubovňa'      -> 'Kamienka, okres Stará Ľubovňa'  SK041A526789


,e_age,e_nuts15,hodnota,soIdLau2,age,soIdLau1
0,14 rokov alebo menej,Bratislava - mestská časť Staré Mesto,7116.0,SK0101528595,AGE_Y_LE14,SK0101
1,65 rokov alebo viac,Bratislava - mestská časť Staré Mesto,10270.0,SK0101528595,AGE_Y_GE65,SK0101
2,Od 15 do 64 rokov,Bratislava - mestská časť Staré Mesto,30510.0,SK0101528595,AGE_Y15-64,SK0101
3,Spolu,Bratislava - mestská časť Staré Mesto,47896.0,SK0101528595,TOTAL,SK0101
4,14 rokov alebo menej,Bratislava - mestská časť Podunajské Biskupice,3180.0,SK0102529311,AGE_Y_LE14,SK0102


### 2b. Harmonise — one row per municipality

Column names follow the legacy table (`soIdLau2`, `soIdLau1`, `municipalityName`, `districtName`)
with the CZ/PL band names `popTotal`, `popAge0to14`, `popAge15to64`, `popAge65over`. Asserted:
every municipality of the item list is present, one value per age, bands sum to `popTotal`.

In [10]:
AGE_COLS = {'TOTAL': 'popTotal', 'AGE_Y_LE14': 'popAge0to14',
            'AGE_Y15-64': 'popAge15to64', 'AGE_Y_GE65': 'popAge65over'}
assert not sk_raw.duplicated(['soIdLau2', 'age']).any(), 'duplicate municipality x age'
sk_new = (sk_raw.pivot(index=['soIdLau2', 'soIdLau1', 'e_nuts15'], columns='age', values='hodnota')
          .rename(columns=AGE_COLS)[list(AGE_COLS.values())].reset_index().rename_axis(columns=None)
          .rename(columns={'e_nuts15': 'municipalityName'}))
sk_new = sk_new.merge(districts.rename(columns={'district': 'soIdLau1'}), on='soIdLau1', how='left')
sk_new = sk_new[['soIdLau2', 'soIdLau1', 'municipalityName', 'districtName', *AGE_COLS.values()]]

# Items of the list that the cube returns no data for (checked, not assumed).
NO_DATA = {'SK0413523551': 'Javorina (vojenský obvod)'}
missing = set(muni.acronym) - set(sk_new.soIdLau2)
assert missing == set(NO_DATA), f'municipalities missing from the chunks: {sorted(missing - set(NO_DATA))}'
assert set(sk_new.soIdLau2) <= set(muni.acronym)
assert sk_new[list(AGE_COLS.values())].notna().all().all(), 'missing values'
assert (sk_new[list(AGE_COLS.values())] % 1 == 0).all().all(), '31 Dec stocks should be whole persons'
sk_new = sk_new.astype({c: 'int64' for c in AGE_COLS.values()})
bsum = sk_new[['popAge0to14', 'popAge15to64', 'popAge65over']].sum(axis=1)
assert (bsum == sk_new.popTotal).all(), 'bands do not add up to the total'
sk_new['ref_year'] = 2025
sk_new['ref_date'] = '2025-12-31'

print(f'sk_new: {len(sk_new):,} rows x {sk_new.shape[1]} cols, ids unique: {sk_new.soIdLau2.is_unique}')
print(f'population total     : {sk_new.popTotal.sum():,}')
print('no data for         :', list(NO_DATA.values()))
print('bands sum to popTotal: all rows')
sk_new.head()

sk_new: 2,927 rows x 10 cols, ids unique: True
population total     : 5,409,407
no data for         : ['Javorina (vojenský obvod)']
bands sum to popTotal: all rows


,soIdLau2,soIdLau1,municipalityName,districtName,popTotal,popAge0to14,popAge15to64,popAge65over,ref_year,ref_date
0,SK0101528595,SK0101,Bratislava - mestská časť Staré Mesto,Bratislava I,47896,7116,30510,10270,2025,2025-12-31
1,SK0102529311,SK0102,Bratislava - mestská časť Podunajské Biskupice,Bratislava II,22998,3180,15029,4789,2025,2025-12-31
2,SK0102529320,SK0102,Bratislava - mestská časť Ružinov,Bratislava II,84233,12887,55062,16284,2025,2025-12-31
3,SK0102529338,SK0102,Bratislava - mestská časť Vrakuňa,Bratislava II,20178,2735,13635,3808,2025,2025-12-31
4,SK0103529346,SK0103,Bratislava - mestská časť Nové Mesto,Bratislava III,45966,7352,30127,8487,2025,2025-12-31


### 2c. Joinability — legacy `soIdLau2` vs the API municipality code, and both vs the workbook

Same id format on both sides, so the raw sets are compared first; then the legacy key rule
(`pop.sk_key`: `soIdLau2` minus its `soIdLau1` prefix) is matched against the 2025 workbook.

In [11]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# old = legacy_raw['public.sk_population_age_groups_lau2_21'][['soIdLau2', 'municipalityName', 'Total']]
# j = old.merge(sk_new[['soIdLau2', 'municipalityName', 'popTotal']], on='soIdLau2', how='outer',
#               suffixes=('_old', '_new'), indicator=True)
# print(j['_merge'].value_counts().to_string())
# print('\nonly in legacy:')
# print(j.loc[j._merge == 'left_only', ['soIdLau2', 'municipalityName_old', 'Total']].to_string(index=False) or '  none')
# print('\nonly in new:')
# print(j.loc[j._merge == 'right_only', ['soIdLau2', 'municipalityName_new', 'popTotal']].to_string(index=False) or '  none')
# both = j[j._merge == 'both']
# renamed = both[both.municipalityName_old.str.strip() != both.municipalityName_new.str.strip()]
# print(f'\nsame id, different name: {len(renamed)}')
# print(renamed[['soIdLau2', 'municipalityName_old', 'municipalityName_new']].head(15).to_string(index=False))

In [12]:
sk_new_lau = sk_new.assign(lau_code=[pop.sk_key(a, b) for a, b in zip(sk_new.soIdLau2, sk_new.soIdLau1)])
print(pop.match_report({COUNTRY: sk_new_lau}, lau).to_string(index=False))

wb = set(lau['lau_code'])
print('\nnew LAU codes not in the 2025 workbook:')
print(sk_new_lau.loc[~sk_new_lau.lau_code.isin(wb), ['lau_code', 'municipalityName', 'popTotal']]
      .to_string(index=False) or '  none')
print('\nworkbook LAU codes not covered by the new file:')
print(lau.loc[~lau.lau_code.isin(sk_new_lau.lau_code), ['lau_code', 'lau_name_national']]
      .to_string(index=False) or '  none')

country  source_rows  workbook_lau  matched  source_unmatched  lau_uncovered
     SK         2927          2927     2927                 0              0

new LAU codes not in the 2025 workbook:
Empty DataFrame
Columns: [lau_code, municipalityName, popTotal]
Index: []

workbook LAU codes not covered by the new file:
Empty DataFrame
Columns: [lau_code, lau_name_national]
Index: []


### 2d. Territorial changes and non-LAU units

The item list also has **Javorina (vojenský obvod)**, a military district that is not a LAU in the
2025 workbook (other military districts, e.g. Valaškovce, are). The cube returns no data for it, so
`SK_NOT_LAU` is only a guard in case a re-issue adds values. `SK_MERGERS` is the same mechanism as
Austria, empty for now. The assert fails on any other code that does not match.

In [13]:
SK_NOT_LAU = {'523551': 'Javorina (vojenský obvod)'}
SK_MERGERS = {}   # new LAU code -> workbook (2025) LAU codes it replaces

dropped = sk_new_lau[sk_new_lau.lau_code.isin(SK_NOT_LAU)]
sk_new_lau = sk_new_lau[~sk_new_lau.lau_code.isin(SK_NOT_LAU)].reset_index(drop=True)
sk_new_lau['replaces_lau'] = sk_new_lau['lau_code'].map(lambda c: ';'.join(SK_MERGERS.get(c, [])) or None)

covered = set(sk_new_lau['lau_code']) | {o for olds in SK_MERGERS.values() for o in olds}
unexplained_new = sorted(set(sk_new_lau['lau_code']) - wb - set(SK_MERGERS))
unexplained_wb = sorted(wb - covered)
assert not unexplained_new and not unexplained_wb, (unexplained_new, unexplained_wb)
print('dropped (not a LAU):')
print(dropped[['lau_code', 'municipalityName', 'popTotal']].to_string(index=False))
print(f'\nnew rows            : {len(sk_new_lau):,}')
print(f'workbook LAU covered: {len(wb & covered):,} / {len(wb):,}  ({len(SK_MERGERS)} mergers)')

dropped (not a LAU):


Empty DataFrame
Columns: [lau_code, municipalityName, popTotal]
Index: []

new rows            : 2,927
workbook LAU covered: 2,927 / 2,927  (0 mergers)


In [14]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# # Same LAU, two vintages.
# cmp = (legacy[['lau_code', 'pop_total', 'y_0_14', 'y_15_64', 'y_65plus']]
#        .merge(sk_new_lau[['lau_code', 'municipalityName', 'popTotal',
#                           'popAge0to14', 'popAge15to64', 'popAge65over']],
#               on='lau_code', how='outer', indicator=True))
# print(cmp['_merge'].value_counts().to_string())
# b = cmp[(cmp._merge == 'both') & (cmp.pop_total > 0)].copy()
# b['diff_%'] = 100 * (b.popTotal - b.pop_total) / b.pop_total
# print(f'\ntotal  legacy {b.pop_total.sum():,.0f}  ->  new {b.popTotal.sum():,.0f}')
# print(b['diff_%'].describe().round(2).to_string())
# print('\nband shares (%), legacy -> new (complete legacy rows):')
# bc = b.dropna(subset=['y_0_14', 'y_15_64', 'y_65plus'])
# for o, n in [('y_0_14', 'popAge0to14'), ('y_15_64', 'popAge15to64'), ('y_65plus', 'popAge65over')]:
#     print(f'  {o:<9} {100 * bc[o].sum() / bc.pop_total.sum():5.1f} -> {100 * bc[n].sum() / bc.popTotal.sum():5.1f}')
# print('\nlegacy rows with a suppressed band, now:')
# inc = legacy.loc[~legacy.band_complete, 'lau_code']
# print(sk_new_lau[sk_new_lau.lau_code.isin(inc)][['lau_code', 'municipalityName', 'popTotal',
#       'popAge0to14', 'popAge15to64', 'popAge65over']].to_string(index=False))
# print('\nlargest relative changes:')
# print(b.reindex(b['diff_%'].abs().sort_values(ascending=False).index)
#        .head(10)[['lau_code', 'municipalityName', 'pop_total', 'popTotal', 'diff_%']].round(1).to_string(index=False))

### 2e. Join to the v4atua LAU subset (GeoJSON)

Slovak features only (`CNTR_CODE = 'SK'`); `GISCO_ID = 'SK_<LAU code>'`.

In [15]:
import geopandas as gpd

LAU_SUBSET = Path('data/eu/ref-lau-2024-01m/subsetv4atua/LAU_RG_01M_2024_4326.geojson')
sk_geo = gpd.read_file(LAU_SUBSET, where="CNTR_CODE = 'SK'")
sk_geo['lau_code'] = sk_geo['GISCO_ID'].str.split('_', n=1).str[1]
assert sk_geo['lau_code'].is_unique
print(f'{LAU_SUBSET.name}: {len(sk_geo):,} SK features')

jg = sk_geo.drop(columns='geometry').merge(
    sk_new_lau[['lau_code', 'soIdLau2', 'municipalityName', 'popTotal']],
    on='lau_code', how='outer', indicator=True)
print('\n' + jg['_merge'].value_counts().to_string())
print('\ngeojson only:', jg.loc[jg._merge == 'left_only', 'lau_code'].tolist())
print('new file only:', jg.loc[jg._merge == 'right_only', ['lau_code', 'municipalityName']].values.tolist())

m = jg[(jg._merge == 'both') & (jg.POP_2024 > 0)].copy()
m['diff_%'] = 100 * (m.popTotal - m.POP_2024) / m.POP_2024
print(f'\npopulation  GISCO POP_2024 {jg.POP_2024.sum():,.0f}  vs  new popTotal {jg.popTotal.sum():,.0f}')
print(m['diff_%'].describe().round(2).to_string())
name_diff = m[m.LAU_NAME.str.strip() != m.municipalityName.str.strip()]
print(f'\nsame code, different name: {len(name_diff)}')
print(name_diff[['lau_code', 'LAU_NAME', 'municipalityName']].head(15).to_string(index=False))
print('\nlargest relative changes:')
print(m.reindex(m['diff_%'].abs().sort_values(ascending=False).index)
       .head(8)[['lau_code', 'LAU_NAME', 'POP_2024', 'popTotal', 'diff_%']].round(1).to_string(index=False))

LAU_RG_01M_2024_4326.geojson: 2,927 SK features

_merge
both          2927
left_only        0
right_only       0

geojson only: []
new file only: []

population  GISCO POP_2024 5,424,687  vs  new popTotal 5,409,407
count    2926.00
mean        0.20
std         3.22
min       -14.29
25%        -1.58
50%        -0.14
75%         1.61
max        45.45

same code, different name: 205
lau_code                        LAU_NAME               municipalityName
  500305  Hosťovce (okres Zlaté Moravce)  Hosťovce, okres Zlaté Moravce
  500607 Nevidzany (okres Zlaté Moravce) Nevidzany, okres Zlaté Moravce
  500674         Podhorany (okres Nitra)         Podhorany, okres Nitra
  500976    Zlatno (okres Zlaté Moravce)    Zlatno, okres Zlaté Moravce
  501069       Brestovec (okres Komárno)       Brestovec, okres Komárno
  501956 Trnávka (okres Dunajská Streda) Trnávka, okres Dunajská Streda
  502065             Beša (okres Levice)             Beša, okres Levice
  502669        Podlužany (okres Levice) 

## 3. Write — common-schema dump

One CSV per country, same columns in all five `00_b*` notebooks, read by
`00_extract_base_data_eu.ipynb` §2c to attach the 2025 population to the LAU subset GeoJSON.

| column | content |
|---|---|
| `lau_code`, `gisco_id` | LAU code (text) and `<CC>_<lau_code>`, the subset key |
| `country`, `lau_name` | country code, national name |
| `pop_total`, `y_0_14`, `y_15_64`, `y_65plus` | persons (`pop.HARMONISED_BANDS`) |
| `band_method`, `ref_year`, `ref_date`, `source` | `exact`; reference year/date; dataset id |
| `native_id` | the source's own key |
| `replaces_lau` | workbook LAU codes a merged row covers (`;`-joined), else empty |
| `includes_native` | native units summed into the row (`;`-joined), else empty |

Skips if the file exists; set `REWRITE_DUMP = True` to overwrite. Read it back with `dtype=str`
for the code columns (HU codes have leading zeros).

In [16]:
REWRITE_DUMP = False
DUMP_COLS = ['lau_code', 'gisco_id', 'country', 'lau_name', 'pop_total', *pop.HARMONISED_BANDS,
             'band_method', 'ref_year', 'ref_date', 'source', 'native_id', 'replaces_lau', 'includes_native']
DUMP_FILE = OUT_DIR / f'lau_population_2025_{COUNTRY.lower()}.csv'

f = sk_new_lau.reset_index(drop=True)
dump = pd.DataFrame({
    'lau_code': f['lau_code'].astype(str),
    'gisco_id': COUNTRY + '_' + f['lau_code'].astype(str),
    'country': COUNTRY,
    'lau_name': f['municipalityName'],
    'pop_total': f['popTotal'], 'y_0_14': f['popAge0to14'], 'y_15_64': f['popAge15to64'], 'y_65plus': f['popAge65over'],
    'band_method': 'exact',
    'ref_year': f['ref_year'], 'ref_date': f['ref_date'],
    'source': 'SUSR_om7006rr',
    'native_id': f['soIdLau2'].astype(str),
    'replaces_lau': f['replaces_lau'],
    'includes_native': None,
})[DUMP_COLS]

assert dump['lau_code'].is_unique and dump['gisco_id'].is_unique
assert (dump[pop.HARMONISED_BANDS].sum(axis=1) == dump['pop_total']).all(), 'bands do not add up'
assert dump[['pop_total', *pop.HARMONISED_BANDS]].notna().all().all()
replaced = {o for r in dump['replaces_lau'].dropna() for o in r.split(';')}
merged_new = set(dump.loc[dump['replaces_lau'].notna(), 'lau_code'])
assert (set(dump['lau_code']) - merged_new) | replaced == set(lau['lau_code']), 'dump does not cover the workbook LAUs'

if DUMP_FILE.exists() and not REWRITE_DUMP:
    print('[exists]', DUMP_FILE, '-- set REWRITE_DUMP = True to overwrite')
else:
    dump.to_csv(DUMP_FILE, index=False, encoding='utf-8')
    print('[wrote] ', DUMP_FILE)
back = pd.read_csv(DUMP_FILE, dtype={'lau_code': str, 'gisco_id': str, 'native_id': str})
assert list(back.columns) == DUMP_COLS and back['lau_code'].tolist() == dump['lau_code'].tolist(), 'file differs from dump'
print(f'{len(back):,} rows, population {back.pop_total.sum():,}; '
      f"merged rows {int(back.replaces_lau.notna().sum())}, aggregated rows {int(back.includes_native.notna().sum())}")
back.head(3)

[exists] data\eu\ref-national-statistical-offices\sk\lau_population_2025_sk.csv -- set REWRITE_DUMP = True to overwrite
2,927 rows, population 5,409,407; merged rows 0, aggregated rows 0


,lau_code,gisco_id,country,lau_name,pop_total,y_0_14,y_15_64,y_65plus,band_method,ref_year,ref_date,source,native_id,replaces_lau,includes_native
0,528595,SK_528595,SK,Bratislava - mestská časť Staré Mesto,47896,7116,30510,10270,exact,2025,2025-12-31,SUSR_om7006rr,SK0101528595,NaN,NaN
1,529311,SK_529311,SK,Bratislava - mestská časť Podunajské Biskupice,22998,3180,15029,4789,exact,2025,2025-12-31,SUSR_om7006rr,SK0102529311,NaN,NaN
2,529320,SK_529320,SK,Bratislava - mestská časť Ružinov,84233,12887,55062,16284,exact,2025,2025-12-31,SUSR_om7006rr,SK0102529320,NaN,NaN
